# Lekce 12 - Snížení historie chatu pomocí agentova scratchpadu

Tento notebook ukazuje, jak spravovat kontext v dlouhých konverzacích pomocí Microsoft Agent Framework. Jak konverzace rostou, zvyšuje se počet tokenů — nakonec přesahují kontextové okno modelu. Tento problém řešíme pomocí **vzorového shrnutí kontextu** a **agentova scratchpadu** pro trvalou paměť.

## Co se naučíte:
1. **Proč je řízení kontextu důležité**: Pochopení limitů tokenů a kontextových oken
2. **Agenti orientovaní na kontext**: Vytváření agentů, kteří spravují vlastní kontext konverzace
3. **Vzorové shrnutí kontextu**: Použití nástrojů ke zhuštění historie konverzace
4. **Agentův scratchpad**: Trvalá paměť, která přežívá redukci kontextu

## Předpoklady:
- Nastavení Azure OpenAI s nakonfigurovanými proměnnými prostředí
- Znalost základních konceptů agentů z předchozích lekcí


## Nastavení


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Proč je správa kontextu důležitá

Každý LLM má omezené **kontextové okno** – maximální počet tokenů, které může zpracovat v jednom požadavku. Jak konverzace na více kol pokračuje:

- **Počet tokenů roste lineárně** s každou uživatelskou zprávou a odpovědí asistenta.
- **Tokeny v promptu dominují nákladům**, protože celá historie se odesílá znovu v každém kole.
- Nakonec konverzace **překročí kontextové okno** a model buď ořízne text, nebo dojde k chybě.

### Strategie pro správu kontextu

| Strategie | Jak funguje | Kompromis |
|---|---|---|
| **Oříznutí** | Vyřadit nejstarší zprávy | Ztráta počátečního kontextu |
| **Shrnutí** | Zkrátit starší zprávy do souhrnu | Některé detaily ztraceny, ale klíčové body zachovány |
| **Scratchpad / externí paměť** | Uložit klíčová fakta mimo konverzaci | Vyžaduje volání nástrojů, ale přežije jakékoli zkrácení |

V tomto sešitu kombinujeme **shrnutí** s **nástrojem scratchpad**, aby agent mohl udržovat kontinuitu i při kondenzaci historie konverzace.


## Vytvoření agenta citlivého na kontext


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## Simulace dlouhého rozhovoru

Projděme si vícetahový rozhovor, abychom viděli, jak se kontext hromadí. Agent by si měl pamatovat klíčové detaily (preference, rozpočet, data cesty) napříč tahy a prokázat kontinuitu.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Všimněte si, jak si agent uchovává kontext z předchozích kol — ví o Japonsku, sushi, chrámech, fotografování, rozpočtu 3000 dolarů, sólovém cestování a dubnovém období. V krátkém rozhovoru to funguje dobře, ale jak se konverzace rozrůstá, odesílání celé historie začíná být nákladné.

Pokračujme v rozhovoru dalším koly, abychom viděli, jak se kontext kumuluje:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Vzor shrnutí kontextu

Jak se konverzace rozrůstá, můžeme použít **nástroj pro shrnutí**, který zhušťuje nasbíraný kontext do kompaktní podoby. Agent tento nástroj volá, aby zaznamenal klíčové preference, takže i když jsou starší zprávy odstraněny, podstatné informace zůstanou zachovány.

Tento vzor je stavebním kamenem pro sofistikovanější redukci historie:
1. Agent identifikuje klíčová fakta z konverzace
2. Zavolá nástroj pro shrnutí, aby je uložil
3. Starší zprávy lze bezpečně odstranit, protože shrnutí zachycuje to, co je důležité

Níže definujeme nástroj `summarize_preferences`, který může agent volat, aby zaznamenal kompaktní shrnutí toho, co se dozvěděl.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Shrnutí

V této lekci jste se naučili, jak spravovat kontext v dlouhotrvajících konverzacích agenta pomocí Microsoft Agent Framework:

### Klíčové koncepty
- **Kontextová okna jsou omezená** — každý token v historii konverzace něco stojí a počítá se do limitu.
- **Nástroje pro shrnutí** umožňují agentovi zkomprimovat nahromaděný kontext do stručných shrnutí, čímž se snižuje používání tokenů při zachování podstatných informací.
- **Poznámkové bloky agenta** poskytují trvalou externí paměť, která přetrvává i přes zkrácení konverzace.

### Co jste vytvořili
- **Kontextově uvědomělý agent**, který udržuje kontinuitu v rámci vícekrokových konverzací
- **Nástroj pro shrnutí** (`summarize_preferences`), který zaznamenává klíčové informace o uživateli v kompaktním formátu
- **Vícekroková konverzace** demonstrující zachování kontextu a zpracování změn

### Praktické použití
- **Zákaznické servisní boti**: Pamatují si preference během dlouhých podporujících sezení
- **Osobní asistenti**: Sledují probíhající projekty bez nutnosti znovu vysvětlovat kontext
- **Vzdělávací lektoři**: Udržují pokrok studenta v průběhu mnoha interakcí

### Další kroky
- Implementovat plnohodnotný nástroj poznámkového bloku s ukládáním do souboru
- Přidat automatické ořezávání historie po shrnutí
- Kombinovat s vektorovými databázemi pro sémantické vyhledávání paměti
- Vytvářet agenty, kteří mohou po dnech s plným kontextem pokračovat v konverzacích


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Prohlášení o omezení odpovědnosti**:
Tento dokument byl přeložen pomocí AI překladatelské služby [Co-op Translator](https://github.com/Azure/co-op-translator). Přestože usilujeme o co největší přesnost, mějte prosím na paměti, že automatizované překlady mohou obsahovat chyby nebo nepřesnosti. Originální dokument v jeho mateřském jazyce by měl být považován za autoritativní zdroj. Pro kritické informace se doporučuje profesionální lidský překlad. Nejsme odpovědní za jakékoli nedorozumění nebo nesprávné interpretace vzniklé použitím tohoto překladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
